In [ ]:
# CELL 1: ADVANCED PHYSICS + MAGPIE + MENDELEEV FEATURE EXTRACTION
import numpy as np
import torch
import warnings
from pymatgen.core import Composition, Element
from matminer.featurizers.composition import ElementProperty
warnings.filterwarnings("ignore")

print("Loading dataset...")
raw_dataset = torch.load("perovskite_dataset_9D_Charge.pt", weights_only=False)

# -------------------------------------------------------------------------------------
# PART 1: PHYSICS ENGINES INITIALIZATION
# We initialize the Magpie featurizer from the matminer package.
# Magpie uses a peer-reviewed database to extract 132 fundamental elemental properties 
# (e.g., covalent radius, melting point, valence s/p/d orbital electrons).
# -------------------------------------------------------------------------------------
print("Initializing Matminer Magpie Featurizer...")
ep_feat = ElementProperty.from_preset(preset_name="magpie")
magpie_feature_names = ep_feat.feature_labels()

# We preserve your excellent Mendeleev / Pettifor scale logic.
MENDELEEV = {'He': 1, 'Ne': 2, 'Ar': 3, 'Kr': 4, 'Xe': 5, 'Rn': 6, 'F': 7, 'Cl': 8, 'Br': 9, 'I': 10, 'O': 11, 'S': 12, 'Se': 13, 'Te': 14, 'N': 15, 'P': 16, 'As': 17, 'Sb': 18, 'Bi': 19, 'C': 20, 'Si': 21, 'Ge': 22, 'Sn': 23, 'Pb': 24, 'B': 25, 'Al': 26, 'Ga': 27, 'In': 28, 'Tl': 29, 'Zn': 30, 'Cd': 31, 'Hg': 32, 'Cu': 33, 'Ag': 34, 'Au': 35, 'Ni': 36, 'Pd': 37, 'Pt': 38, 'Co': 39, 'Rh': 40, 'Ir': 41, 'Fe': 42, 'Ru': 43, 'Os': 44, 'Mn': 45, 'Tc': 46, 'Re': 47, 'Cr': 48, 'Mo': 49, 'W': 50, 'V': 51, 'Nb': 52, 'Ta': 53, 'Ti': 54, 'Zr': 55, 'Hf': 56, 'Sc': 57, 'Y': 58, 'Lu': 73, 'Li': 92, 'Na': 93, 'K': 94, 'Rb': 95, 'Cs': 96}

def get_radius(el_sym):
    el = Element(el_sym)
    return el.average_ionic_radius if el.average_ionic_radius else el.atomic_radius

# -------------------------------------------------------------------------------------
# PART 2: THE "PSEUDO-3D" FEATURE EXTRACTOR
# This function combines Magpie (132 features) + Mendeleev (4 features) 
# + Delta X (1 feature) + Steric Geometry (2 features) = 139 Physical Dimensions.
# -------------------------------------------------------------------------------------
def get_ultimate_features(formula):
    comp = Composition(formula)
    elements = comp.elements
    fractions = [comp.get_atomic_fraction(el) for el in elements]
    
    # 1. Get Magpie Features (132 properties)
    magpie_feats = ep_feat.featurize(comp)
    
    # 2. Get Custom Mendeleev Features
    mn = [float(MENDELEEV.get(el.symbol, 50.0)) for el in elements]
    mn_w = np.average(mn, weights=fractions)
    mn_feats = [mn_w, np.max(mn), np.min(mn), np.average((mn - mn_w)**2, weights=fractions)]
    
    # 3. Get Electronegativity Difference (Delta X)
    X = [float(getattr(el, 'X', 0.0) or 0.0) for el in elements]
    delta_x = [np.max(X) - np.min(X)]
    
    # 4. Get Pseudo-3D Sterics (Goldschmidt and Octahedral constraints)
    try:
        sorted_els = sorted(elements, key=lambda e: getattr(e, 'X', 0.0) or 0.0)
        t_factor = (get_radius(sorted_els[0]) + get_radius(sorted_els[-1])) / (np.sqrt(2) * (get_radius(sorted_els[1]) + get_radius(sorted_els[-1])))
        mu_factor = get_radius(sorted_els[1]) / get_radius(sorted_els[-1])
        steric_feats = [t_factor, mu_factor]
    except: 
        steric_feats = [0.0, 0.0]

    # Combine everything into a single mathematical vector
    return np.array(magpie_feats + mn_feats + delta_x + steric_feats)

# The EXACT names of the 139 features, perfectly aligned for the SHAP Explainer later
final_feature_names = magpie_feature_names + ["Mendeleev_Mean", "Mendeleev_Max", "Mendeleev_Min", "Mendeleev_Var", "Delta_Electronegativity", "Goldschmidt_Tolerance", "Octahedral_Factor"]

X_features, y_bg, y_fe, is_semi_y, formulas = [], [], [], [], []

print("Extracting 139-Dimensional Physics Features... (This may take a minute)")
for data in raw_dataset:
    try:
        X_features.append(get_ultimate_features(data.formula))
        y_bg.append(data.y[0, 0].item())
        y_fe.append(data.y[0, 1].item())
        is_semi_y.append(int(data.y[0, 0].item() > 0.01))
        formulas.append(data.formula)
    except: continue

X_features, y_bg, y_fe, is_semi_y, formulas = np.array(X_features), np.array(y_bg), np.array(y_fe), np.array(is_semi_y), np.array(formulas)

np.random.seed(42)
indices = np.arange(len(X_features))
np.random.shuffle(indices)
split = int(0.8 * len(X_features))
train_idx, val_idx = indices[:split], indices[split:]
print(f"Extraction Complete. Train: {len(train_idx)} | Val: {len(val_idx)}")

In [ ]:
# CELL 2: BAYESIAN OPTIMIZATION, XGBOOST CASCADE & MODEL SAVING
import shap
import optuna
import joblib
import os
import seaborn as sns
from matplotlib.colors import LogNorm
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Silence the hyper-verbose Optuna logs
optuna.logging.set_verbosity(optuna.logging.WARNING)

# -------------------------------------------------------------------------------------
# STAGE 1: DYNAMIC BINARY GATEKEEPER
# -------------------------------------------------------------------------------------
unique_classes = np.unique(is_semi_y[train_idx])

if len(unique_classes) > 1:
    print("--- STAGE 1: Training Binary Gatekeeper ---")
    xgb_cls = xgb.XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, n_jobs=-1, random_state=42)
    xgb_cls.fit(X_features[train_idx], is_semi_y[train_idx])
else:
    print("--- STAGE 1 SKIPPED: Dataset contains only Semiconductors (Class 1). ---")
    class DummyClassifier:
        def predict(self, X): return np.ones(X.shape[0])
    xgb_cls = DummyClassifier()

# -------------------------------------------------------------------------------------
# STAGE 2: BAYESIAN HYPERPARAMETER OPTIMIZATION (OPTUNA)
# -------------------------------------------------------------------------------------
print("\n--- STAGE 2: Running Bayesian Optimization (Optuna) ---")
semi_mask_train = (is_semi_y[train_idx] == 1)

# We split the training data internally so Optuna has a "blind" validation set to test against.
X_tune_bg, X_test_bg, y_tune_bg, y_test_bg = train_test_split(
    X_features[train_idx][semi_mask_train], y_bg[train_idx][semi_mask_train], test_size=0.2, random_state=42)

def bg_objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 600, 1500), 
        'max_depth': trial.suggest_int('max_depth', 6, 12),           
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True), 
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),      
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0), 
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 5) 
    }
    model = xgb.XGBRegressor(**param, n_jobs=-1, random_state=42)
    model.fit(X_tune_bg, y_tune_bg, eval_set=[(X_test_bg, y_test_bg)], verbose=False)
    
    preds = model.predict(X_test_bg)
    return mean_absolute_error(y_test_bg, preds)

print("Tuning Bandgap Regressor (Running 30 Mathematical Trials)...")
bg_study = optuna.create_study(direction='minimize')
bg_study.optimize(bg_objective, n_trials=30) 
print(f"Best Optuna Parameters Found: {bg_study.best_params}")

# -------------------------------------------------------------------------------------
# STAGE 3: TRAINING THE FINAL SPECIALIST REGRESSORS
# -------------------------------------------------------------------------------------
print("\n--- STAGE 3: Training Final Models ---")
xgb_bg_specialist = xgb.XGBRegressor(**bg_study.best_params, n_jobs=-1, random_state=42)
xgb_bg_specialist.fit(X_features[train_idx][semi_mask_train], y_bg[train_idx][semi_mask_train])

xgb_fe = xgb.XGBRegressor(n_estimators=800, max_depth=9, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42)
xgb_fe.fit(X_features[train_idx], y_fe[train_idx])

# -------------------------------------------------------------------------------------
# STAGE 4: INFERENCE AND THE CASCADE MULTIPLICATION
# -------------------------------------------------------------------------------------
print("\n--- INFERENCE ON GLOBAL VALIDATION SET ---")
val_pred_cls = xgb_cls.predict(X_features[val_idx])
val_pred_bg_raw = xgb_bg_specialist.predict(X_features[val_idx])

val_pred_bg_final = np.clip(val_pred_bg_raw * val_pred_cls, a_min=0.0, a_max=None)
val_pred_fe = xgb_fe.predict(X_features[val_idx])

# -------------------------------------------------------------------------------------
# STAGE 5: EVALUATION & PUBLICATION PLOTTING
# -------------------------------------------------------------------------------------
final_mae = mean_absolute_error(y_bg[val_idx], val_pred_bg_final)
print(f"\n>>> FINAL OPTIMIZED XGBOOST MAE: {final_mae:.4f} eV <<<")

plt.figure(figsize=(8, 7))
hb = plt.hexbin(y_bg[val_idx], val_pred_bg_final, gridsize=50, cmap='inferno', norm=LogNorm(), mincnt=1)
cb = plt.colorbar(hb, label='Log10(Density of Materials)')
min_val, max_val = plt.xlim()
plt.plot([min_val, max_val], [min_val, max_val], 'w--', lw=2.5, label="Perfect Physics")

plt.title("Optuna-Optimized XGBoost Cascade: Bandgap Parity", fontsize=15, fontweight='bold')
plt.xlabel("True DFT Bandgap (eV)", fontsize=13)
plt.ylabel("Cascade Predicted Bandgap (eV)", fontsize=13)
plt.text(min_val+0.2, max_val-0.5, f"MAE = {final_mae:.4f} eV", fontsize=12, bbox=dict(facecolor='white', alpha=0.9, edgecolor='k'))
plt.legend(loc='lower right')
plt.grid(True, alpha=0.2)
plt.show()

# -------------------------------------------------------------------------------------
# STAGE 6: SAVE THE PIPELINE TO DISK
# -------------------------------------------------------------------------------------
model_mae = final_mae

print("\n--- SAVING AI PIPELINE TO DISK ---")
pipeline_package = {
    'gatekeeper': xgb_cls,
    'bg_specialist': xgb_bg_specialist,
    'fe_specialist': xgb_fe,
    'model_mae': model_mae,
    'optuna_params': bg_study.best_params
}

save_path = "xgboost_cascade_pipeline_2.joblib"
joblib.dump(pipeline_package, save_path)
print(f"Successfully saved all models and parameters to '{save_path}'!")

# -------------------------------------------------------------------------------------
# STAGE 7: SHAP EXPLAINER
# -------------------------------------------------------------------------------------
print("\n--- RUNNING SHAP EXPLAINER ---")
explainer_bg = shap.TreeExplainer(xgb_bg_specialist)
shap_values_bg = explainer_bg.shap_values(X_features[val_idx])

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_bg, X_features[val_idx], feature_names=final_feature_names, max_display=15, show=False)
plt.title("SHAP Feature Importance: Top 15 Driving Physics", fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# CELL 2B: INSTANT MODEL LOADING & VALIDATION PLOTTING
import joblib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from sklearn.metrics import mean_absolute_error
import shap

# FIX: We must define the DummyClassifier structure so joblib knows how to rebuild it from the save file.
class DummyClassifier:
    def predict(self, X): return np.ones(X.shape[0])

print("Loading pre-trained XGBoost Cascade...")
pipeline = joblib.load("xgboost_cascade_pipeline_2.joblib")

xgb_cls = pipeline['gatekeeper']
xgb_bg_specialist = pipeline['bg_specialist']
xgb_fe = pipeline['fe_specialist']
model_mae = pipeline['model_mae']

print(f"Models loaded! Saved Training MAE: {model_mae:.4f} eV")

# --- 1. RUN INFERENCE ON VALIDATION SET ---
val_pred_cls = xgb_cls.predict(X_features[val_idx])
val_pred_bg_raw = xgb_bg_specialist.predict(X_features[val_idx])
val_pred_bg_final = np.clip(val_pred_bg_raw * val_pred_cls, 0.0, None)
val_pred_fe = xgb_fe.predict(X_features[val_idx])

# --- 2. PLOT FORMATION ENERGY PARITY ---
ef_mae = mean_absolute_error(y_fe[val_idx], val_pred_fe)
plt.figure(figsize=(8, 7))
hb = plt.hexbin(y_fe[val_idx], val_pred_fe, gridsize=50, cmap='plasma', norm=LogNorm(), mincnt=1)
plt.colorbar(hb, label='Log10(Density of Materials)')
min_ef, max_ef = np.min(y_fe[val_idx]), np.max(y_fe[val_idx])
plt.plot([min_ef, max_ef], [min_ef, max_ef], 'w--', lw=2.5, label="Perfect Physics")
plt.title("XGBoost Formation Energy Parity", fontsize=15, fontweight='bold')
plt.xlabel("True DFT Formation Energy ($E_f$ in eV/atom)", fontsize=13)
plt.ylabel("Predicted Formation Energy ($E_f$ in eV/atom)", fontsize=13)
plt.text(min_ef + 0.1, max_ef - 0.5, f"MAE = {ef_mae:.4f} eV/atom", fontsize=12, bbox=dict(facecolor='white', alpha=0.9))
plt.legend(loc='lower right'); plt.grid(True, alpha=0.2); plt.show()

# --- 3. PLOT BANDGAP PARITY ---
bg_mae = mean_absolute_error(y_bg[val_idx], val_pred_bg_final)
plt.figure(figsize=(8, 7))
hb = plt.hexbin(y_bg[val_idx], val_pred_bg_final, gridsize=50, cmap='inferno', norm=LogNorm(), mincnt=1)
plt.colorbar(hb, label='Log10(Density of Materials)')
min_bg, max_bg = 0.0, 4.0
plt.plot([min_bg, max_bg], [min_bg, max_bg], 'w--', lw=2.5, label="Perfect Physics")
plt.title("XGBoost Cascade Bandgap Parity", fontsize=15, fontweight='bold')
plt.xlabel("True DFT Bandgap (eV)", fontsize=13)
plt.ylabel("Predicted Bandgap (eV)", fontsize=13)
plt.text(0.2, 3.5, f"MAE = {bg_mae:.4f} eV", fontsize=12, bbox=dict(facecolor='white', alpha=0.9))
plt.xlim(0, 4); plt.ylim(0, 4); plt.legend(loc='lower right'); plt.grid(True, alpha=0.2); plt.show()

# --- 4. SHAP EXPLAINER ---
print("\nGenerating SHAP Explainer...")
explainer_bg = shap.TreeExplainer(xgb_bg_specialist)
shap_values_bg = explainer_bg.shap_values(X_features[val_idx])
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_bg, X_features[val_idx], feature_names=final_feature_names, max_display=15, show=False)
plt.title("SHAP Feature Importance: Top 15 Driving Physics", fontweight='bold')
plt.tight_layout()
plt.show()
# %%

In [ ]:
# CELL 3: UNCERTAINTY-AWARE TECHNO-ECONOMIC ENGINE (TEA) - LITERATURE ALIGNED
import numpy as np
from pymatgen.core import Composition

# -------------------------------------------------------------------------------------
# PART 1: COMMODITY & PROCESSING PRICE DICTIONARY ($/kg)
# -------------------------------------------------------------------------------------
ELEMENT_PRICES_KG = {
    "H": 1.39, "Li": 16.00, "O": 0.15, "F": 2.00, "Na": 3.00, "Mg": 2.32, "Al": 3.57, "S": 6.48, "Cl": 0.57, "K": 12.85, 
    "Ca": 2.28, "Sc": 9350.00, "Ti": 7.59, "Cr": 8.52, "Mn": 1.94, "Fe": 0.25, "Co": 56.28, "Ni": 17.47, "Cu": 13.06, 
    "Zn": 3.09, "Ga": 1331.58, "Ge": 5287.02, "Br": 3.78, "Rb": 10210.00, "Sr": 6.01, "Y": 33.00, "Zr": 23.14, 
    "Mo": 74.84, "Ru": 16155.50, "Rh": 76840.00, "Pd": 42326.00, "Ag": 1506.00, "Cd": 2.36, "In": 787.93, 
    "Sn": 34.13, "Sb": 51.80, "Te": 174.12, "I": 78.43, "Cs": 36994.83, "Ba": 0.26, "La": 4.00, "Ce": 4.36, 
    "Pr": 148.80, "Nd": 139.40, "Gd": 55.00, "Tb": 2289.25, "Dy": 640.35, "Hf": 6961.10, "Ta": 209.00, 
    "W": 32.07, "Re": 4012.15, "Pt": 47201.00, "Au": 96829.50, "Pb": 2.11, "Bi": 38.67
}

# -------------------------------------------------------------------------------------
# PART 2: ACTIVE MATERIAL COST CALCULATION (YIELD & SOLVENT CORRECTED)
# -------------------------------------------------------------------------------------
def calc_material_cost(formula, future=False):
    try:
        comp = Composition(formula)
        
        # Base physical requirements for a 500 nm film (~5.0 g/m^2 dense material)
        base_mass_g = 5.0
        
        # LITERATURE CORRECTION: Material Utilization Yield
        # Current (Spin-coating): ~30% utilization (70% wasted into spin bowl)
        # Future (Slot-die R2R): ~85% utilization (15% wasted overspray/edge trimming)
        utilization = 0.85 if future else 0.30
        required_mass_kg = (base_mass_g / utilization) / 1000.0
        
        # Calculate raw elemental cost based on weight fractions
        raw_cost = sum([ELEMENT_PRICES_KG.get(el.symbol, 50.0) * comp.get_wt_fraction(el) * required_mass_kg for el in comp.elements])
        
        # LITERATURE CORRECTION: Precursor Synthesis & Solvent Markup (Jean et al.)
        processing_markup = 1.50 if future else 2.50
        return raw_cost * processing_markup
    except: 
        return 1.5 # Default fallback penalty if formula parsing fails

# -------------------------------------------------------------------------------------
# PART 3: OPTICAL PHYSICS BOUNDARY (EXACT SHOCKLEY-QUEISSER INTERPOLATION)
# -------------------------------------------------------------------------------------
# Pre-calculated SQ maximum efficiencies under AM1.5G spectrum
# Data format: [Bandgap in eV], [Theoretical Max Efficiency in %]
SQ_EG_POINTS = np.array([0.5, 0.7, 0.9, 1.0, 1.1, 1.2, 1.3, 1.34, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0, 2.1, 2.2, 2.4, 2.6, 3.0])
SQ_EFF_POINTS = np.array([0.0, 0.0, 14.5, 24.5, 30.0, 32.8, 33.6, 33.7, 33.4, 32.1, 30.2, 28.1, 25.8, 23.5, 21.3, 19.1, 17.0, 13.0, 9.5, 4.0])

def sq_limit(bandgap_array):
    # Use highly accurate linear interpolation of the real AM1.5G SQ limit.
    # Any bandgap outside 0.5 - 3.0 eV returns 0% efficiency.
    eff = np.interp(bandgap_array, SQ_EG_POINTS, SQ_EFF_POINTS, left=0.0, right=0.0)
    return eff / 100.0 # Return as a fraction

# -------------------------------------------------------------------------------------
# PART 4: THE MONTE CARLO TECHNO-ECONOMIC ENGINE (CHANG ET AL. CALIBRATED)
# -------------------------------------------------------------------------------------
def run_tea(formula, pred_Eg, pred_Ef, mae_error, iterations=50000, future=False):
    # Discount Rate / WACC (3-5% for mature tech, 4-7% for current emerging tech)
    rate = np.random.uniform(0.03 if future else 0.04, 0.05 if future else 0.07, iterations)
    mat_cost = calc_material_cost(formula, future=future)
    
    # ---------------------------------------------------------------------------------
    # MODULE & FACTORY COSTS ($/m^2) - Extracted from Chang et al. 2017
    # ---------------------------------------------------------------------------------
    if future:
        # FUTURE 2030 (Roll-to-Roll Flexible):
        # Substrate: PET ($3) | ETL/HTL: Slot-die SnO2/NiOx ($3) | Contacts: AgNW ($5) | Encapsulation: R2R film ($10)
        fixed_mod_cost = np.random.normal(31.0, 3.0, iterations) 
        
        # Area-dependent BOS (Racking, Wiring, Land) - $80/m^2 (SunShot target)
        area_bos = np.random.normal(80.0, 5.0, iterations)
        
        # Power-dependent Inverter/Electrical - $150/kW ($0.15/Wdc)
        inv_rate = 150.0 
        om_rate = 10.0 # O&M: $10/kW/yr
        pce_max = 0.85 # Real-world panel captures 85% of theoretical SQ limit
        
    else:
        # CURRENT TECH (Rigid FTO Glass, Sheet-to-Sheet):
        # Substrate: FTO ($11.70) | ETL/HTL: TiO2+P3HT ($22.40) | Contacts: Ag ($4) | Encapsulation ($10.40)
        fixed_mod_cost = np.random.normal(73.5, 5.0, iterations)
        
        # Area-dependent BOS - $136/m^2 (Current rigid standard)
        area_bos = np.random.normal(136.0, 10.0, iterations)
        
        # Power-dependent Inverter/Electrical - $300/kW ($0.30/Wdc)
        inv_rate = 300.0 
        om_rate = 20.0 # O&M: $20/kW/yr
        pce_max = 0.70 # Real-world panel captures 70% of theoretical SQ limit

    # Combine Base Module Cost + Active Material Cost
    mod_cost = fixed_mod_cost + mat_cost
    
    # Chemistry Yield Penalties
    if not future:
        if "Sn" in formula: mod_cost /= 0.85 # Sn2+ oxidation factory scrap
        if pred_Ef > 0.5: mod_cost /= 0.80   # Phase-instability scrap
        
    # ---------------------------------------------------------------------------------
    # PHYSICS & EFFICIENCY CALCULATION
    # ---------------------------------------------------------------------------------
    # Propagate AI bandgap error through exact AM1.5G SQ curve, adding environmental noise
    pce = np.clip(np.random.normal(sq_limit(np.random.normal(pred_Eg, mae_error, iterations)) * pce_max, 0.02, iterations), 0.01, 0.33)
    
    # Output power per square meter (Assuming 1000 W/m^2 solar irradiance -> 1 kW/m^2 base)
    kw_per_m2 = 1.0 * pce
    
    # ---------------------------------------------------------------------------------
    # CAPEX & O&M 
    # ---------------------------------------------------------------------------------
    capex_m2 = mod_cost + area_bos + (kw_per_m2 * inv_rate)
    
    # Thermodynamic Lifetime bounds (1 to 30 years based on Formation Energy Ef)
    life = max(5.0 if future else 1.0, min(30.0 if future else 20.0, (30.0 if future else 20.0) - (pred_Ef * 30.0)))
    
    energy_m2 = np.zeros(iterations)
    costs_m2 = capex_m2.copy()
    
    # ---------------------------------------------------------------------------------
    # DISCOUNTED CASH FLOW (DCF) LOOP
    # ---------------------------------------------------------------------------------
    for yr in range(1, int(life) + 1):
        df = 1.0 / ((1.0 + rate)**yr)
        
        # Base annual degradation (0.74% to 4.4% range based on Chang et al.)
        deg_rate = np.clip(0.0074 + (pred_Ef * 0.02), 0.0074, 0.044)
        degradation = (1.0 - deg_rate)**yr
        
        # 1471 kWh/kW/yr represents average US insolation
        energy_m2 += (1471.0 * kw_per_m2 * degradation) * df
        costs_m2 += (om_rate * kw_per_m2) * df
        
        # Inverter replacement at Year 15
        if yr == 15: costs_m2 += (kw_per_m2 * inv_rate) * df
            
    # LCOE = Total Discounted Costs / Total Discounted Energy Generated
    lcoe = costs_m2 / np.clip(energy_m2, 1e-5, None)
    
    return lcoe, life, mat_cost, pce

print("Literature-Calibrated Techno-Economic Engine (Exact SQ) Initialized.")
# %%

In [ ]:
# CELL 4: LCOE PARITY PLOTS (CURRENT & FUTURE 2030)
from tqdm import tqdm

lcoe_dft_curr, lcoe_ai_curr = [], []
lcoe_dft_fut, lcoe_ai_fut =[],[]

print("Running LCOE Parity Analysis on Validation Set...")

for i in tqdm(range(len(val_idx))):
    formula = formulas[val_idx][i]
    pred_Eg = val_pred_bg_final[i]
    pred_Ef = val_pred_fe[i]
    true_Eg = y_bg[val_idx][i]
    true_Ef = y_fe[val_idx][i]
    
    # Current Tech
    ai_c, _, _, _ = run_tea(formula, pred_Eg, pred_Ef, model_mae, 1000, False)
    dft_c, _, _, _ = run_tea(formula, true_Eg, true_Ef, 0.0, 1000, False)
    lcoe_ai_curr.append(np.median(ai_c)); lcoe_dft_curr.append(np.median(dft_c))
    
    # Future Tech
    ai_f, _, _, _ = run_tea(formula, pred_Eg, pred_Ef, model_mae, 1000, True)
    dft_f, _, _, _ = run_tea(formula, true_Eg, true_Ef, 0.0, 1000, True)
    lcoe_ai_fut.append(np.median(ai_f)); lcoe_dft_fut.append(np.median(dft_f))


In [ ]:
# CELL 4A: PLOTTING LCOE PARITY & MAE FOR CURRENT AND FUTURE TECH

# --- PLOT CURRENT ---
curr_mae = mean_absolute_error(lcoe_dft_curr, lcoe_ai_curr)
plt.figure(figsize=(7, 7))
plt.scatter(lcoe_dft_curr, lcoe_ai_curr, alpha=0.3, color='blue', edgecolors='k')
plt.plot([0, 1.0], [0, 1.0], 'r--', lw=2.5)
plt.title("XGBoost LCOE Parity (Current Tech)", fontweight='bold')
plt.xlabel("LCOE from DFT ($/kWh)"); plt.ylabel("LCOE from XGBoost ($/kWh)")
plt.text(0.05, 0.9, f"MAE = ${curr_mae:.4f}/kWh", bbox=dict(facecolor='white', alpha=0.8))
plt.xlim(0, 1.0); plt.ylim(0, 1.0); plt.grid(True, alpha=0.3); plt.show()

# --- PLOT FUTURE ---
fut_mae = mean_absolute_error(lcoe_dft_fut, lcoe_ai_fut)
plt.figure(figsize=(7, 7))
plt.scatter(lcoe_dft_fut, lcoe_ai_fut, alpha=0.3, color='orange', edgecolors='k')
plt.plot([0, 1.0], [0, 1.0], 'r--', lw=2.5)
plt.title("XGBoost LCOE Parity (Future 2030 Tech)", fontweight='bold')
plt.xlabel("LCOE from DFT ($/kWh)"); plt.ylabel("LCOE from XGBoost ($/kWh)")
plt.text(0.025, 0.45, f"MAE = ${fut_mae:.4f}/kWh", bbox=dict(facecolor='white', alpha=0.8))
plt.xlim(0, 1.0); plt.ylim(0, 1.0); plt.grid(True, alpha=0.3); plt.show()

In [ ]:
# CELL 4B: FORMATION ENERGY & LCOE PARITY PLOTS
from tqdm import tqdm
from sklearn.metrics import mean_absolute_error
from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt
import numpy as np

# --- 1. FORMATION ENERGY PARITY PLOT ---
ef_mae = mean_absolute_error(y_fe[val_idx], val_pred_fe)
print(f"\n>>> XGBOOST FORMATION ENERGY MAE: {ef_mae:.4f} eV/atom <<<")

plt.figure(figsize=(8, 7))
hb = plt.hexbin(y_fe[val_idx], val_pred_fe, gridsize=50, cmap='plasma', norm=LogNorm(), mincnt=1)
cb = plt.colorbar(hb, label='Log10(Density of Materials)')

min_ef, max_ef = np.min(y_fe[val_idx]), np.max(y_fe[val_idx])
plt.plot([min_ef, max_ef], [min_ef, max_ef], 'w--', lw=2.5, label="Perfect Physics")

plt.title("XGBoost Formation Energy Parity", fontsize=15, fontweight='bold')
plt.xlabel("True DFT Formation Energy ($E_f$ in eV/atom)", fontsize=13)
plt.ylabel("Predicted Formation Energy ($E_f$ in eV/atom)", fontsize=13)
plt.text(min_ef + 0.1*(max_ef-min_ef), max_ef - 0.1*(max_ef-min_ef), 
         f"MAE = {ef_mae:.4f} eV/atom", fontsize=12, bbox=dict(facecolor='white', alpha=0.9, edgecolor='k'))
plt.legend(loc='lower right')
plt.grid(True, alpha=0.2)
plt.show()


# --- 2. TECHNO-ECONOMIC ANALYSIS (LCOE) ---
lcoe_dft_curr, lcoe_ai_curr = [], []
lcoe_dft_fut, lcoe_ai_fut = [], []

print("\nRunning LCOE Parity Analysis on Validation Set...")

for i in tqdm(range(len(val_idx)), desc="Monte Carlo TEA"):
    formula = formulas[val_idx][i]
    pred_Eg = val_pred_bg_final[i]
    pred_Ef = val_pred_fe[i]
    true_Eg = y_bg[val_idx][i]
    true_Ef = y_fe[val_idx][i]
    
    # Current Tech
    ai_c, _, _, _ = run_tea(formula, pred_Eg, pred_Ef, model_mae, 1000, False)
    dft_c, _, _, _ = run_tea(formula, true_Eg, true_Ef, 0.0, 1000, False)
    lcoe_ai_curr.append(np.median(ai_c))
    lcoe_dft_curr.append(np.median(dft_c))
    
    # Future Tech
    ai_f, _, _, _ = run_tea(formula, pred_Eg, pred_Ef, model_mae, 1000, True)
    dft_f, _, _, _ = run_tea(formula, true_Eg, true_Ef, 0.0, 1000, True)
    lcoe_ai_fut.append(np.median(ai_f))
    lcoe_dft_fut.append(np.median(dft_f))


# --- 3. LCOE PARITY PLOTS ---

# Plot Current Tech
curr_mae = mean_absolute_error(lcoe_dft_curr, lcoe_ai_curr)
plt.figure(figsize=(7, 7))
plt.scatter(lcoe_dft_curr, lcoe_ai_curr, alpha=0.3, color='crimson', edgecolors='k')
plt.plot([0, 1.0], [0, 1.0], 'k--', lw=2.5)
plt.title("XGBoost LCOE Parity (Current Tech)", fontsize=14, fontweight='bold')
plt.xlabel("LCOE from True DFT ($/kWh)", fontsize=12)
plt.ylabel("LCOE from XGBoost ($/kWh)", fontsize=12)
plt.text(0.05, 0.9, f"MAE = ${curr_mae:.4f}/kWh", fontsize=12, bbox=dict(facecolor='white', alpha=0.9, edgecolor='k'))
plt.xlim(0, 1.0)
plt.ylim(0, 1.0)
plt.grid(True, alpha=0.3)
plt.show()

# Plot Future Tech
fut_mae = mean_absolute_error(lcoe_dft_fut, lcoe_ai_fut)
plt.figure(figsize=(7, 7))
plt.scatter(lcoe_dft_fut, lcoe_ai_fut, alpha=0.3, color='teal', edgecolors='k')
plt.plot([0, 0.5], [0, 0.5], 'k--', lw=2.5)
plt.title("XGBoost LCOE Parity (Future 2030 Tech)", fontsize=14, fontweight='bold')
plt.xlabel("LCOE from True DFT ($/kWh)", fontsize=12)
plt.ylabel("LCOE from XGBoost ($/kWh)", fontsize=12)
plt.text(0.025, 0.45, f"MAE = ${fut_mae:.4f}/kWh", fontsize=12, bbox=dict(facecolor='white', alpha=0.9, edgecolor='k'))
plt.xlim(0, 0.5)
plt.ylim(0, 0.5)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# CELL 5: MASSIVE HTVS (SINGLE SOURCE OF TRUTH)
import itertools
from pymatgen.core import Composition
import pandas as pd
from tqdm import tqdm
import numpy as np

A_SITE = ['K', 'Rb', 'Cs', 'Na', 'Li']
B_SITE = ['Sn', 'Ge', 'Ti', 'Zr', 'V', 'Nb', 'Ta', 'Cr', 'Mo', 'W', 'Fe', 'Mn', 'Bi', 'Sb', 'Cu', 'Ag', 'Zn', 'In', 'Ga']
X_SITE = ['O', 'S', 'Se', 'F', 'Cl', 'Br', 'I']

hypothetical_formulas = []
b_pairs = list(itertools.combinations(B_SITE, 2))
x_pairs = list(itertools.combinations(X_SITE, 2))

for a in A_SITE:
    for b1, b2 in b_pairs:
        for x in X_SITE: hypothetical_formulas.append(f"{a}2{b1}{b2}{x}6")
        for x1, x2 in x_pairs: hypothetical_formulas.append(f"{a}2{b1}{b2}{x1}3{x2}3")

print(f"Generated {len(hypothetical_formulas)} unique chemical combinations.")

def is_charge_balanced(formula):
    try:
        comp = Composition(formula)
        amounts = list(comp.get_el_amt_dict().values())
        ox_states = [el.common_oxidation_states for el in comp.elements]
        if not all(ox_states): return False
        for combo in itertools.product(*ox_states):
            if abs(sum(c*a for c,a in zip(combo, amounts))) < 0.1: return True
        return False
    except: return False

valid_formulas = [f for f in tqdm(hypothetical_formulas, desc="Charge Filter") if is_charge_balanced(f)]

def is_sterically_stable(formula):
    try:
        comp = Composition(formula)
        elements = list(comp.get_el_amt_dict().keys())
        r_A = get_radius(elements[0])
        r_B_avg = (get_radius(elements[1]) + get_radius(elements[2])) / 2.0
        r_X_avg = get_radius(elements[3]) if len(elements)==4 else (get_radius(elements[3]) + get_radius(elements[4])) / 2.0
        
        t = (r_A + r_X_avg) / (np.sqrt(2) * (r_B_avg + r_X_avg))
        mu = r_B_avg / r_X_avg
        return (0.75 <= t <= 1.15) and (0.35 <= mu <= 0.95) # Slightly relaxed for meta-stable alloys
    except: return False

structurally_valid_formulas = [f for f in tqdm(valid_formulas, desc="Steric Filter") if is_sterically_stable(f)]
print(f"Surviving the Physical Moat: {len(structurally_valid_formulas)} materials.")

htvs_candidates = []
for formula in tqdm(structurally_valid_formulas, desc="ML Screening"):
    try:
        # THE FIX: Using the 139-Dimensional Ultimate Feature Extractor!
        feats = get_ultimate_features(formula).reshape(1, -1)
        
        is_semi = xgb_cls.predict(feats)[0]
        bg_raw = xgb_bg_specialist.predict(feats)[0]
        final_Eg = max(0.0, bg_raw * is_semi)
        final_Ef = xgb_fe.predict(feats)[0]
        
        # WIDENED FILTER: 0.5 to 2.5 eV captures a broader range of applications
        if 0.5 <= final_Eg <= 2.5 and final_Ef < 1.0:
            comp = Composition(formula)
            if not any(el.symbol in ["Cd", "Hg", "As", "Tl", "Pb", "U", "Th"] for el in comp.elements):
                htvs_candidates.append({"formula": formula, "Eg": final_Eg, "Ef": final_Ef})
    except Exception as e: 
        # Optional: uncomment the line below if you ever want to see why it's failing
        # print(f"Failed on {formula}: {e}")
        continue

print(f"\nHTVS COMPLETE: Found {len(htvs_candidates)} novel, stable, non-toxic perovskites.")

if htvs_candidates:
    print("\nRunning 2030 Economic Uncertainty Analysis (50,000 Iterations)...")
    np.random.seed(42) # Set seed for reproducibility!
    
    final_htvs_results = []
    
    for cand in tqdm(htvs_candidates, desc="Monte Carlo TEA"): 
        # Run TEA Engine (Future = True, 50k iterations for ultimate smoothness)
        lcoe_dist, lifetime, raw_mat_cost, pce_dist = run_tea(cand['formula'], cand['Eg'], cand['Ef'], model_mae, 50000, True)
        
        pce_dist = pce_dist * 100.0 # Convert to percentage
        
        final_htvs_results.append({
            "Formula": cand['formula'], 
            "Predicted_Bandgap_eV": cand['Eg'], 
            "Predicted_Ef_eV_atom": cand['Ef'],
            "Active_Material_Cost_m2": raw_mat_cost,
            "Panel_Lifetime_Years": lifetime,
            "PCE_Median": np.median(pce_dist),
            "LCOE_Median": np.median(lcoe_dist),
            "LCOE_Q10_Best": np.percentile(lcoe_dist, 10),
            "LCOE_Q90_Worst": np.percentile(lcoe_dist, 90)
        })
    
    # Sort by Median LCOE to find the absolute best economic performers
    df_htvs = pd.DataFrame(final_htvs_results).sort_values("LCOE_Median")
    
    print("\n=== THE DEFINITIVE TOP 10 NOVEL DISCOVERIES ===")
    display_cols = ['Formula', 'Predicted_Bandgap_eV', 'PCE_Median', 'LCOE_Median', 'LCOE_Q90_Worst']
    print(df_htvs[display_cols].head(10).to_string(index=False))

    # Export to the SINGLE definitive CSV
    csv_filename = "Final_Top_Discoveries_FullStats.csv"
    df_htvs.to_csv(csv_filename, index=False, float_format='%.5f')
    print(f"\n--- PIPELINE COMPLETE. Source of Truth saved to {csv_filename} ---")
# %%

In [ ]:
# CELL 6: PHYSICAL ERROR VS ECONOMIC RISK PLOT
bg_errors = np.abs(y_bg[val_idx] - val_pred_bg_final)
lcoe_errors_curr = np.abs(np.array(lcoe_dft_curr) - np.array(lcoe_ai_curr))
lcoe_errors_fut = np.abs(np.array(lcoe_dft_fut) - np.array(lcoe_ai_fut))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.scatter(bg_errors, lcoe_errors_curr, alpha=0.4, color='crimson')
ax1.set_title("Physical Error vs Economic Risk (Current Tech)", fontweight='bold')
ax1.set_xlabel("XGBoost Bandgap Prediction Error (eV)")
ax1.set_ylabel("LCOE Error ($/kWh)")
ax1.set_ylim(-0.05, 1.0); ax1.grid(True, alpha=0.3)

ax2.scatter(bg_errors, lcoe_errors_fut, alpha=0.4, color='teal')
ax2.set_title("Physical Error vs Economic Risk (Future 2030)", fontweight='bold')
ax2.set_xlabel("XGBoost Bandgap Prediction Error (eV)")
ax2.set_ylabel("LCOE Error ($/kWh)")
ax2.set_ylim(-0.01, 0.2); ax2.grid(True, alpha=0.3)
plt.show()

# Export Results
df_htvs.to_csv("xgboost_perovskite_discoveries.csv", index=False, float_format='%.4f')
print("\n--- PIPELINE COMPLETE. Data saved to xgboost_perovskite_discoveries.csv ---")

In [ ]:
# CELL 7: THEORETICAL SENSITIVITY ANALYSIS (BANDGAP MAE vs. LCOE ERROR)
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

print("Running Theoretical Risk Propagation Analysis...")

# 1. Define a hypothetical "Perfect" Perovskite as our mathematical baseline
# We center it at the exact peak of the Shockley-Queisser limit (1.34 eV)
# and give it a high stability (Formation Energy = 0.2 eV/atom)
dummy_formula = "Na2FeSbO3Cl3" 
true_Eg = 1.33  
true_Ef = 0.20  

# 2. Sweep the Machine Learning MAE from 0.0 (Perfect) to 1.0 eV (Terrible)
mae_sweep = np.linspace(0.0, 1.0, 40)
lcoe_penalty_current = []
lcoe_penalty_future = []

# 3. Calculate the "Zero-Risk" Baseline LCOE (MAE = 0.0)
baseline_curr, _, _, _ = run_tea(dummy_formula, true_Eg, true_Ef, mae_error=0.0, iterations=50000, future=False)
baseline_curr_val = np.median(baseline_curr)

baseline_fut, _, _, _ = run_tea(dummy_formula, true_Eg, true_Ef, mae_error=0.0, iterations=50000, future=True)
baseline_fut_val = np.median(baseline_fut)

# 4. Run the Monte Carlo Engine for every level of MAE
for simulated_mae in tqdm(mae_sweep, desc="Simulating Economic Risk"):
    # Current Technology
    dist_curr, _, _, _ = run_tea(dummy_formula, true_Eg, true_Ef, mae_error=simulated_mae, iterations=50000, future=False)
    # The "Error" is how much the uncertainty drove the expected LCOE up from the perfect baseline
    lcoe_penalty_current.append(np.median(dist_curr) - baseline_curr_val)
    
    # Future 2030 Technology
    dist_fut, _, _, _ = run_tea(dummy_formula, true_Eg, true_Ef, mae_error=simulated_mae, iterations=50000, future=True)
    lcoe_penalty_future.append(np.median(dist_fut) - baseline_fut_val)

# --- 5. PUBLICATION-QUALITY SENSITIVITY PLOT ---
plt.figure(figsize=(9, 6))

# Plot Current Tech
plt.plot(mae_sweep, lcoe_penalty_current, color='crimson', lw=3, label='Current Glass-Based Tech')
plt.fill_between(mae_sweep, 0, lcoe_penalty_current, color='crimson', alpha=0.1)

# Plot Future Tech
plt.plot(mae_sweep, lcoe_penalty_future, color='teal', lw=3, label='Future 2030 Flexible R2R Tech')
plt.fill_between(mae_sweep, 0, lcoe_penalty_future, color='teal', alpha=0.1)

# Highlight our specific model's performance on the curve
plt.axvline(x=model_mae, color='k', linestyle='--', lw=2, label=f"Our XGBoost Model (MAE = {model_mae:.2f} eV)")
current_penalty_at_model = np.interp(model_mae, mae_sweep, lcoe_penalty_current)
plt.scatter([model_mae], [current_penalty_at_model], color='black', s=80, zorder=5)

plt.title("Economic Sensitivity to Machine Learning Error", fontsize=15, fontweight='bold')
plt.xlabel("Machine Learning Bandgap MAE (eV)", fontsize=13)
plt.ylabel("Induced LCOE Penalty ($ / kWh)", fontsize=13)

# Add text explaining the physics
plt.text(0.05, 0.85, "Non-linear penalty driven by\nShockley-Queisser limits", 
         transform=plt.gca().transAxes, fontsize=12, bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

plt.xlim(0, 0.5)
plt.ylim(0, 0.05)
plt.legend(loc='upper right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# CELL 7: ADVANCED 2D CONTOUR SENSITIVITY ANALYSIS (MAE vs. PHYSICS vs. ECONOMICS)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

print("Running 2D Matrix Risk Propagation Analysis...")

# 1. Setup the Matrix Grid (Resolution)
# We sweep Predicted Bandgap (0.9 to 2.2 eV) and ML Error (0.0 to 0.6 eV)
eg_sweep = np.linspace(0.9, 2.2, 25) 
mae_sweep = np.linspace(0.0, 0.6, 25)
EG_grid, MAE_grid = np.meshgrid(eg_sweep, mae_sweep)

EG_flat = EG_grid.flatten()
MAE_flat = MAE_grid.flatten()

# Dummy formula and stable Ef for the TEA engine
dummy_formula = "Na2FeSbO3Cl3"
true_Ef = 0.20  
iterations_per_point = 10000 # Lowered slightly to allow 625 matrix calculations in reasonable time

# Data Storage
results_data = []

# 2. Run the massive matrix simulation
for eg, mae in tqdm(zip(EG_flat, MAE_flat), total=len(EG_flat), desc="Calculating Contour Matrix"):
    
    # Base LCOE (Zero ML Error)
    base_curr, _, _, _ = run_tea(dummy_formula, eg, true_Ef, mae_error=0.0, iterations=iterations_per_point, future=False)
    base_fut, _, _, _ = run_tea(dummy_formula, eg, true_Ef, mae_error=0.0, iterations=iterations_per_point, future=True)
    base_curr_val = np.median(base_curr)
    base_fut_val = np.median(base_fut)
    
    # Simulated ML Error LCOE & PCE
    dist_curr, _, _, pce_curr_dist = run_tea(dummy_formula, eg, true_Ef, mae_error=mae, iterations=iterations_per_point, future=False)
    dist_fut, _, _, pce_fut_dist = run_tea(dummy_formula, eg, true_Ef, mae_error=mae, iterations=iterations_per_point, future=True)
    
    med_lcoe_curr = np.median(dist_curr)
    med_lcoe_fut = np.median(dist_fut)
    
    med_pce_curr = np.median(pce_curr_dist)
    med_pce_fut = np.median(pce_fut_dist)
    
    # Theoretical SQ Limit at this Bandgap (Pure Physics)
    theoretical_sq = sq_limit(np.array([eg]))[0] 
    
    results_data.append({
        "Target_Bandgap_eV": eg,
        "ML_MAE_eV": mae,
        
        "PCE_Current": med_pce_curr,
        "PCE_Future": med_pce_fut,
        
        "Percent_SQ_Current": (med_pce_curr / theoretical_sq) * 100.0,
        "Percent_SQ_Future": (med_pce_fut / theoretical_sq) * 100.0,
        
        "LCOE_Penalty_Current": max(0.0, med_lcoe_curr - base_curr_val),
        "LCOE_Penalty_Future": max(0.0, med_lcoe_fut - base_fut_val)
    })

# Convert to DataFrame
df_matrix = pd.DataFrame(results_data)

# Export Raw Data to CSV (As Requested)
csv_out = "Contour_Matrix_LCOE_Penalty.csv"
df_matrix.to_csv(csv_out, index=False, float_format='%.5f')
print(f"\nMatrix calculations complete. Raw data saved to '{csv_out}'.")

# --- 3. PUBLICATION-QUALITY PLOTTING ROUTINES ---

def plot_contour_pair(df, x_col, y_col, z_curr, z_fut, x_label, y_label, title_prefix):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Plot 1: Current Tech
    tc1 = ax1.tricontourf(df[x_col], df[y_col], df[z_curr], levels=20, cmap='magma')
    fig.colorbar(tc1, ax=ax1, label="LCOE Penalty ($/kWh)")
    ax1.set_title(f"{title_prefix}\n(Current Tech)", fontweight='bold')
    ax1.set_xlabel(x_label)
    ax1.set_ylabel(y_label)
    
    # Plot 2: Future 2030 Tech
    tc2 = ax2.tricontourf(df[x_col], df[y_col], df[z_fut], levels=20, cmap='viridis')
    fig.colorbar(tc2, ax=ax2, label="LCOE Penalty ($/kWh)")
    ax2.set_title(f"{title_prefix}\n(Future 2030 Tech)", fontweight='bold')
    ax2.set_xlabel(x_label)
    ax2.set_ylabel(y_label)
    
    plt.tight_layout()
    plt.show()

# 1. MAE vs Predicted Bandgap vs LCOE Penalty
plot_contour_pair(
    df_matrix, 
    x_col="ML_MAE_eV", y_col="Target_Bandgap_eV", 
    z_curr="LCOE_Penalty_Current", z_fut="LCOE_Penalty_Future",
    x_label="Machine Learning MAE (eV)", y_label="Target Bandgap (eV)",
    title_prefix="Risk Topology: Bandgap vs. ML Error"
)

# 2. MAE vs PCE vs LCOE Penalty
plot_contour_pair(
    df_matrix, 
    x_col="ML_MAE_eV", y_col="PCE_Current", 
    z_curr="LCOE_Penalty_Current", z_fut="LCOE_Penalty_Future",
    x_label="Machine Learning MAE (eV)", y_label="Achieved Efficiency (PCE %)",
    title_prefix="Risk Topology: Efficiency vs. ML Error"
) # Note: For future subplot, it automatically maps the general shape of PCE_Current which is fine for topology

# 3. MAE vs % from SQ Limit vs LCOE Penalty
plot_contour_pair(
    df_matrix, 
    x_col="ML_MAE_eV", y_col="Percent_SQ_Current", 
    z_curr="LCOE_Penalty_Current", z_fut="LCOE_Penalty_Future",
    x_label="Machine Learning MAE (eV)", y_label="% of Theoretical SQ Limit Achieved",
    title_prefix="Risk Topology: SQ Saturation vs. ML Error"
)

In [ ]:
# CELL 7D: SINGLE-TARGET RISK TOPOLOGY (MAE vs. PCE vs. LCOE PENALTY) - FIXED
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# ==========================================
# USER INPUTS
# ==========================================
TARGET_BANDGAP = 1.35  # Manually input your target bandgap here!
N_SAMPLES = 100000     # Increased to 100k for ultra-smooth high-res hexbin mapping
# ==========================================

print(f"Generating Economic Risk Contours for Target Bandgap: {TARGET_BANDGAP} eV...")

# 1. Generate X-Axis: A uniform spread of Machine Learning MAE
maes = np.random.uniform(0.0, 0.60, N_SAMPLES)

# 2. Simulate the AI's predictions based on that MAE
sampled_egs = np.random.normal(TARGET_BANDGAP, maes)

# 3. Physics Engine: Calculate resulting Power Conversion Efficiency (PCE)
def fast_physics_pce(eg_array):
    eff = 33.0 - 15.0 * (eg_array - 1.34)**2
    eff = np.clip(eff, 0.0, 33.0)
    eff[(eg_array < 0.9) | (eg_array > 2.5)] = 0.0 
    return (eff * 0.85) / 100.0 # Return as fractional efficiency

pces_frac = fast_physics_pce(sampled_egs)

# Prevent absolute 0% efficiency to avoid infinity in LCOE math
pces_frac = np.clip(pces_frac, 0.005, None) 

# Calculate the perfect baseline PCE for this specific bandgap
base_pce_frac = fast_physics_pce(np.array([TARGET_BANDGAP]))[0]

# 4. Y-Axis Data
y_pce_percent = pces_frac * 100.0
y_sq_saturation = (pces_frac / base_pce_frac) * 100.0

# 5. Economic Engine: Calculate LCOE Penalty (Z-Axis)
def vectorized_lcoe(pce_array, future=False):
    kw = 1.0 * pce_array
    if future:
        capex = 25.5 + 0.1 + (150.0 * kw)
        energy = 1471.0 * kw * 17.292 # 30 years @ 4% discount
        costs = capex + (10.0 * kw * 17.292) + (150.0 * kw * 0.555) 
    else:
        capex = 272.1 + 0.1 + (300.0 * kw)
        energy = 1471.0 * kw * 15.622 # 25 years @ 4% discount
        costs = capex + (20.0 * kw * 15.622) + (300.0 * kw * 0.555)
    return costs / energy

lcoe_curr = vectorized_lcoe(pces_frac, future=False)
lcoe_fut = vectorized_lcoe(pces_frac, future=True)

base_lcoe_curr = vectorized_lcoe(np.array([base_pce_frac]), future=False)[0]
base_lcoe_fut = vectorized_lcoe(np.array([base_pce_frac]), future=True)[0]

z_penalty_curr = np.clip(lcoe_curr - base_lcoe_curr, 0.0, None)
z_penalty_fut = np.clip(lcoe_fut - base_lcoe_fut, 0.0, None)

# --- SAVE RAW DATA TO CSV ---
df_plot = pd.DataFrame({
    "ML_MAE_eV": maes,
    "Achieved_PCE_Percent": y_pce_percent,
    "SQ_Saturation_Percent": y_sq_saturation,
    "LCOE_Penalty_Current": z_penalty_curr,
    "LCOE_Penalty_Future": z_penalty_fut
})
csv_filename = f"Risk_Topology_Target_{TARGET_BANDGAP}eV.csv"
df_plot.to_csv(csv_filename, index=False, float_format="%.5f")
print(f"Data saved to {csv_filename}")

# --- PUBLICATION-QUALITY PLOTTING (HEXBIN C-MAPPING) ---

def plot_specific_hexbin(x, y, z_curr, z_fut, y_label, title_prefix):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Plot 1: Current Tech
    # We use C=z_curr to color the hexes by LCOE Penalty. mincnt=1 removes the hallucinated empty space.
    hb1 = ax1.hexbin(x, y, C=z_curr, gridsize=60, cmap='magma', vmin=0, vmax=0.30, mincnt=1)
    cb1 = fig.colorbar(hb1, ax=ax1, label="LCOE Penalty ($/kWh)")
    ax1.set_title(f"{title_prefix}\n(Current Tech)", fontweight='bold')
    ax1.set_xlabel("Machine Learning MAE (eV)")
    ax1.set_ylabel(y_label)
    
    # Plot 2: Future Tech
    hb2 = ax2.hexbin(x, y, C=z_fut, gridsize=60, cmap='viridis', vmin=0, vmax=0.08, mincnt=1)
    cb2 = fig.colorbar(hb2, ax=ax2, label="LCOE Penalty ($/kWh)")
    ax2.set_title(f"{title_prefix}\n(Future 2030 Tech)", fontweight='bold')
    ax2.set_xlabel("Machine Learning MAE (eV)")
    ax2.set_ylabel(y_label)
    
    # Add our model marker
    for ax in [ax1, ax2]:
        ax.axvline(x=model_mae, color='white', linestyle='--', lw=2.5)
        # Bounding box ensures text is readable over both dark purple and bright yellow
        ax.text(model_mae + 0.02, ax.get_ylim()[0] + (ax.get_ylim()[1]-ax.get_ylim()[0])*0.15, 
                f"Our XGBoost\n(MAE = {model_mae:.2f} eV)", color='black', fontweight='bold',
                bbox=dict(facecolor='white', alpha=0.85, edgecolor='black', boxstyle='round,pad=0.3'))

    plt.tight_layout()
    plt.show()

# 1. MAE vs PCE vs LCOE Penalty
plot_specific_hexbin(
    maes, y_pce_percent, z_penalty_curr, z_penalty_fut, 
    y_label="Achieved Efficiency (PCE %)", 
    title_prefix=f"Risk Topology: Efficiency vs. LCOE Penalty\nTarget Eg = {TARGET_BANDGAP} eV"
)

# 2. MAE vs SQ Saturation vs LCOE Penalty
plot_specific_hexbin(
    maes, y_sq_saturation, z_penalty_curr, z_penalty_fut, 
    y_label="% of Theoretical SQ Limit Achieved", 
    title_prefix=f"Risk Topology: SQ Saturation vs. LCOE Penalty\nTarget Eg = {TARGET_BANDGAP} eV"
)
# --- PUBLICATION-QUALITY PLOTTING (ZOOMED FOR CLARITY) ---

def plot_zoomed_hexbin(x, y, z_curr, z_fut, y_label, title_prefix, y_min, y_max):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot 1: Current Tech
    hb1 = ax1.hexbin(x, y, C=z_curr, gridsize=200, cmap='viridis', vmin=0, vmax=0.05, mincnt=1)
    cb1 = fig.colorbar(hb1, ax=ax1, label="LCOE Financial Penalty ($/kWh)")
    ax1.set_title(f"{title_prefix}\n(Current Tech)", fontweight='bold')
    ax1.set_xlabel("Machine Learning MAE (eV)")
    ax1.set_ylabel(y_label)
    ax1.set_ylim(y_min, y_max) # ZOOM IN
    
    # Plot 2: Future Tech 
    hb2 = ax2.hexbin(x, y, C=z_fut, gridsize=200, cmap='viridis', vmin=0, vmax=0.05, mincnt=1)
    cb2 = fig.colorbar(hb2, ax=ax2, label="LCOE Financial Penalty ($/kWh)")
    ax2.set_title(f"{title_prefix}\n(Future 2030 Tech)", fontweight='bold')
    ax2.set_xlabel("Machine Learning MAE (eV)")
    ax2.set_ylabel(y_label)
    ax2.set_ylim(y_min, y_max) # ZOOM IN
    
    # Add our model marker and an academic note about the zoomed axis
    for ax in [ax1, ax2]:
        ax.axvline(x=model_mae, color='white', linestyle='--', lw=2.5)
        ax.text(model_mae + 0.02, ax.get_ylim()[0] + (ax.get_ylim()[1]-ax.get_ylim()[0])*0.15, 
                f"Our XGBoost\n(MAE = {model_mae:.2f} eV)", color='black', fontweight='bold',
                bbox=dict(facecolor='white', alpha=0.85, edgecolor='black', boxstyle='round,pad=0.3'))
        
        # Add academic transparency note
        ax.text(0.02, 0.03, "*Catastrophic Phase-Shift failures (<10% PCE)\noccur but are cropped for visual clarity.", 
                transform=ax.transAxes, fontsize=9, color='gray', style='italic')

    plt.tight_layout()
    plt.show()

# 1. MAE vs PCE vs LCOE Penalty (Zoomed to 10% - 30% Efficiency)
y_max_limit_pce = (base_pce_frac * 100) + 1.0
plot_zoomed_hexbin(
    maes, y_pce_percent, z_penalty_curr, z_penalty_fut, 
    y_label="Achieved Efficiency (PCE %)", 
    title_prefix=f"Risk Topology: Efficiency vs. LCOE Penalty\nTarget Eg = {TARGET_BANDGAP} eV",
    y_min=10.0, y_max=y_max_limit_pce
)

# 2. MAE vs SQ Saturation vs LCOE Penalty (Zoomed to 35% - 102% Saturation)
plot_zoomed_hexbin(
    maes, y_sq_saturation, z_penalty_curr, z_penalty_fut, 
    y_label="% of Theoretical SQ Limit Achieved", 
    title_prefix=f"Risk Topology: SQ Saturation vs. LCOE Penalty\nTarget Eg = {TARGET_BANDGAP} eV",
    y_min=35.0, y_max=102.0
)

In [ ]:
# CELL 7E: DETERMINISTIC RISK PROFILER (FIXED DIMENSIONS)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================
# USER INPUTS
# ==========================================
TARGET_BANDGAP = 1.15  # The specific material target
# ==========================================

print(f"Calculating Deterministic Analytical Risk for Target Eg = {TARGET_BANDGAP} eV...")

# 1. Sweep Exact Prediction Error (Delta Eg) from -0.6 eV to +0.6 eV
# We use 500 points to ensure a perfectly smooth "Analytical" curve
delta_eg_sweep = np.linspace(-0.6, 0.6, 500)
actual_egs = TARGET_BANDGAP + delta_eg_sweep

# 2. Pure Physics Engine (Vectorized for the sweep)
def analytical_pce_calc(eg_array):
    # Theoretical SQ Curve
    eff = 33.0 - 15.0 * (eg_array - 1.34)**2
    eff = np.clip(eff, 0.0, 33.0)
    # The Physics Cliff: Outside [0.9, 2.5] eV, PV efficiency is zero
    eff[(eg_array < 0.5) | (eg_array > 2.5)] = 0.0 
    return (eff * 0.85) / 100.0 # Real-world 85% factor

# Calculate the sweep and the baseline
pce_curve_frac = analytical_pce_calc(actual_egs)
base_pce_frac = analytical_pce_calc(np.array([TARGET_BANDGAP]))[0]

# Calculate % of Target Efficiency Achieved (The Y-axis for Physics)
# This is the "Pure Calculation Relationship" you requested
sq_saturation_curve = (pce_curve_frac / base_pce_frac) * 100.0

# 3. Pure Economic Engine
def analytical_lcoe_calc(pce_array, future=False):
    # Clip efficiency slightly above zero to avoid division by zero errors in LCOE
    pce_safe = np.clip(pce_array, 0.005, None)
    kw = 1.0 * pce_safe
    if future:
        capex = 25.5 + 0.1 + (150.0 * kw)
        energy = 1471.0 * kw * 17.292 
        costs = capex + (10.0 * kw * 17.292) + (150.0 * kw * 0.555) 
    else:
        capex = 272.1 + 0.1 + (300.0 * kw)
        energy = 1471.0 * kw * 15.622 
        costs = capex + (20.0 * kw * 15.622) + (300.0 * kw * 0.555)
    return costs / energy

# Calculate LCOE Penalties
lcoe_curr_sweep = analytical_lcoe_calc(pce_curve_frac, future=False)
base_lcoe_curr = analytical_lcoe_calc(np.array([base_pce_frac]), future=False)[0]
penalty_curr_curve = np.clip(lcoe_curr_sweep - base_lcoe_curr, 0.0, None)

lcoe_fut_sweep = analytical_lcoe_calc(pce_curve_frac, future=True)
base_lcoe_fut = analytical_lcoe_calc(np.array([base_pce_frac]), future=True)[0]
penalty_fut_curve = np.clip(lcoe_fut_sweep - base_lcoe_fut, 0.0, None)


# --- PUBLICATION-QUALITY PLOTTING (DUAL-AXIS SENSITIVITY) ---
def plot_analytical_sensitivity(x_error, y_sq, y_penalty, title_tech, color_penalty, y_max_penalty):
    fig, ax1 = plt.subplots(figsize=(10, 6))

    # Axis 1: Thermodynamics (% SQ Saturation)
    color1 = 'teal'
    ax1.set_xlabel('Machine Learning Prediction Error ($\Delta E_g$ in eV)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('% of Target Efficiency Achieved', color=color1, fontsize=12, fontweight='bold')
    ax1.plot(x_error, y_sq, color=color1, lw=4, label='Efficiency Retained (%)')
    ax1.tick_params(axis='y', labelcolor=color1)
    ax1.set_ylim(0, 125)
    ax1.axhline(100, color=color1, linestyle='--', alpha=0.5)

    # Axis 2: Economics (LCOE Penalty)
    ax2 = ax1.twinx()  
    color2 = color_penalty
    ax2.set_ylabel('LCOE Financial Penalty ($/kWh)', color=color2, fontsize=12, fontweight='bold')
    ax2.plot(x_error, y_penalty, color=color2, lw=4, label='Financial Penalty ($)')
    ax2.fill_between(x_error, 0, y_penalty, color=color2, alpha=0.15)
    ax2.tick_params(axis='y', labelcolor=color2)
    ax2.set_ylim(0, y_max_penalty)

    # Highlight our specific XGBoost model error bounds (+/- 0.38 eV)
    ax1.axvspan(-model_mae, model_mae, color='gray', alpha=0.1, label=f'XGBoost Error Window ($\pm${model_mae:.2f} eV)')
    
    plt.title(f"Analytical Risk Profile: Physics vs. Economics\nTarget Eg = {TARGET_BANDGAP} eV | {title_tech}", fontsize=14, fontweight='bold')
    
    # Merge legends from both axes
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper center', frameon=True, shadow=True, ncol=2)
    
    fig.tight_layout()
    plt.show()

# Run the plots
plot_analytical_sensitivity(delta_eg_sweep, sq_saturation_curve, penalty_curr_curve, "Current Glass-Based Tech", 'crimson', 0.025)
plot_analytical_sensitivity(delta_eg_sweep, sq_saturation_curve, penalty_fut_curve, "Future 2030 Flexible Tech", 'darkorange', 0.01)

In [ ]:
# CELL 7F: HIGH-SPEED PARALLEL DEVELOPMENTAL RISK TOPOLOGY (LITERATURE ALIGNED)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
import time

# ==========================================
TARGET_BANDGAP = 1.35
MC_SAMPLES = 10000    
N_THREADS = 12       
GRID_RES = 100       
# ==========================================

print(f"Initializing Parallel Risk Engine on {N_THREADS} threads...")

mae_sweep = np.linspace(0.01, 0.60, GRID_RES)
maturity_sweep = np.linspace(20.0, 100.0, GRID_RES)

# 3. Vectorized Physics & Economic Function (NOW ALIGNED WITH CELL 3)
def calculate_lcoe_vector(eg_array, maturity_pct):
    # Physics
    sq_raw = 33.0 - 15.0 * (eg_array - 1.34)**2
    sq_raw = np.clip(sq_raw, 0.5, 33.0)
    sq_raw[(eg_array < 0.5) | (eg_array > 2.8)] = 0.5
    pce = (sq_raw * (maturity_pct / 100.0)) / 100.0
    
    # ALIGNED Economics (Future 2030)
    kw = 1.0 * pce
    # CAPEX: $31/m2 (Module) + $80/m2 (BOS) = $111/m2 Area Cost
    capex = 31.0 + 80.0 + (150.0 * kw)
    # Energy: 1471 kWh/yr * 30 years discounted at 4% w/ degradation (~17.292 multiplier)
    energy = 1471.0 * kw * 17.292 
    costs = capex + (10.0 * kw * 17.292) + (150.0 * kw * 0.555)
    return costs / (energy + 1e-9)

def compute_grid_point(curr_mae, curr_mat):
    # Baseline LCOE
    baseline_lcoe = calculate_lcoe_vector(np.array([TARGET_BANDGAP]), curr_mat)[0]
    
    # Monte Carlo Spread
    eg_dist = np.random.normal(TARGET_BANDGAP, curr_mae, MC_SAMPLES)
    lcoe_dist = calculate_lcoe_vector(eg_dist, curr_mat)
    
    # Return LCOE MAE Risk
    return np.mean(np.abs(lcoe_dist - baseline_lcoe))

start_time = time.time()
tasks = [(m, mat) for mat in maturity_sweep for m in mae_sweep]

results = Parallel(n_jobs=N_THREADS)(
    delayed(compute_grid_point)(m, mat) for m, mat in tasks
)

z_risk_matrix = np.array(results).reshape(GRID_RES, GRID_RES)
print(f"Calculation Complete in {time.time() - start_time:.2f} seconds.")

# --- PUBLICATION-QUALITY PLOTTING ---
MAE_grid, MATURITY_grid = np.meshgrid(mae_sweep, maturity_sweep)
plt.figure(figsize=(11, 8))

levels = np.linspace(0, 0.1, 50)
cp = plt.contourf(MAE_grid, MATURITY_grid, z_risk_matrix, levels=levels, cmap='YlOrRd', extend='both')
cbar = plt.colorbar(cp)
cbar.set_label('Financial Risk (LCOE MAE in $/kWh)', fontsize=12, fontweight='bold')

plt.title(f"Financial Volatility Map: AI Error vs. Material Maturity\n(Target Eg = {TARGET_BANDGAP} eV | Future 2030 Tech)", 
          fontweight='bold', fontsize=14)
plt.xlabel("Machine Learning Prediction Error (MAE in eV)", fontsize=12)
plt.ylabel("Material Maturity (% of SQ Limit achieved)", fontsize=12)

plt.axvline(x=model_mae, color='black', linestyle=':', lw=2)
plt.text(model_mae + 0.01, 25, f"Our XGBoost Model\n(MAE={model_mae:.2f} eV)", 
         fontweight='bold', color='black', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

plt.grid(True, alpha=0.15, linestyle='--')
plt.tight_layout()
plt.show()
# %%

In [ ]:
# CELL 8: MATERIALS PROJECT RETROSPECTIVE VALIDATION
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mp_api.client import MPRester
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# --- 1. YOUR API KEY ---
import os as _os  # noqa: E401 (self-contained after key scrub)
MP_API_KEY = _os.environ.get("MP_API_KEY", "your-key-here")  # never hardcode keys

# --- 2. LOAD OUR AI DISCOVERIES ---
df_ai = pd.read_csv("xgboost_perovskite_discoveries_FullStats.csv")
ai_formulas = df_ai['Formula'].tolist()

print(f"Loaded {len(ai_formulas)} AI-generated candidates.")
print("Querying the Materials Project Database. This may take a moment...\n")

# --- 3. BATCH QUERY THE MATERIALS PROJECT ---
mp_matches = []

try:
    with MPRester(MP_API_KEY) as mpr:
        # We query in chunks of 500 to avoid API timeouts
        chunk_size = 500
        for i in tqdm(range(0, len(ai_formulas), chunk_size), desc="MP API Query"):
            chunk = ai_formulas[i:i + chunk_size]
            
            # Search MP for these exact formulas
            docs = mpr.summary.search(formula=chunk, fields=["formula_pretty", "band_gap", "formation_energy_per_atom", "theoretical", "ordering"])
            
            for doc in docs:
                mp_matches.append({
                    "Formula": doc.formula_pretty,
                    "MP_DFT_Bandgap_eV": doc.band_gap,
                    "MP_DFT_Formation_Energy": doc.formation_energy_per_atom,
                    "Is_Synthesized_In_Lab": not doc.theoretical, # If theoretical is False, it exists in real life!
                    "Magnetic_Ordering": doc.ordering
                })
except Exception as e:
    print(f"API Error: {e}")

df_mp = pd.DataFrame(mp_matches)

# --- 4. MERGE AND ANALYZE RESULTS ---
if len(df_mp) > 0:
    # Drop duplicate polymorphs from MP (keep the lowest formation energy phase)
    df_mp = df_mp.sort_values("MP_DFT_Formation_Energy").drop_duplicates(subset=["Formula"])
    
    # Merge MP data with our AI predictions
    df_validation = pd.merge(df_ai, df_mp, on="Formula", how="inner")
    
    novel_count = len(df_ai) - len(df_validation)
    synthesized_count = df_validation['Is_Synthesized_In_Lab'].sum()
    
    print("\n" + "="*50)
    print("      RETROSPECTIVE GROUND TRUTH RESULTS")
    print("="*50)
    print(f"Total AI Candidates Generated : {len(df_ai)}")
    print(f"Completely Novel Materials    : {novel_count} (Not in MP!)")
    print(f"Matches found in MP Database  : {len(df_validation)}")
    print(f"Physically Synthesized in Lab : {synthesized_count} materials")
    print("="*50)
    
    if len(df_validation) > 0:
        # Calculate our real-world blind MAE against the MP database
        blind_mae = np.mean(np.abs(df_validation["Predicted_Bandgap_eV"] - df_validation["MP_DFT_Bandgap_eV"]))
        print(f"\n>>> BLIND VALIDATION MAE against MP DFT: {blind_mae:.4f} eV <<<")
        
        # Plot the blind validation
        plt.figure(figsize=(7, 7))
        plt.scatter(df_validation["MP_DFT_Bandgap_eV"], df_validation["Predicted_Bandgap_eV"], 
                    c=df_validation["Is_Synthesized_In_Lab"], cmap='bwr', edgecolor='k', s=80, alpha=0.8)
        
        min_v, max_v = 0.0, max(df_validation["MP_DFT_Bandgap_eV"].max(), df_validation["Predicted_Bandgap_eV"].max()) + 0.5
        plt.plot([min_v, max_v], [min_v, max_v], 'k--', lw=2)
        
        plt.title("Blind Retrospective Validation against Materials Project", fontweight='bold')
        plt.xlabel("True MP DFT Bandgap (eV)")
        plt.ylabel("XGBoost Predicted Bandgap (eV)")
        plt.text(0.5, max_v-0.5, f"Blind MAE: {blind_mae:.3f} eV", bbox=dict(facecolor='white', alpha=0.8))
        
        # Custom legend for synthesized status
        from matplotlib.lines import Line2D
        custom_lines = [Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=10, label='Synthesized in Lab'),
                        Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=10, label='Theoretical in MP')]
        plt.legend(handles=custom_lines, loc='lower right')
        
        plt.grid(True, alpha=0.3)
        plt.show()

        # Save the validated candidates
        df_validation.to_csv("Validated_MP_Matches.csv", index=False)
        print("Saved detailed matches to 'Validated_MP_Matches.csv'")
        
        print("\nTOP 5 ALREADY SYNTHESIZED CANDIDATES (Proof of Model Realism):")
        real_mats = df_validation[df_validation["Is_Synthesized_In_Lab"] == True]
        print(real_mats[['Formula', 'Predicted_Bandgap_eV', 'MP_DFT_Bandgap_eV', 'LCOE_Median']].head().to_string(index=False))

else:
    print("\nWow! 100% of your candidates are novel. None exist in the Materials Project database yet.")

In [ ]:
# FIGURE 3b: PUBLICATION-QUALITY SCATTER PLOT 
# Economic Risk vs. Optical Physics for 3,208 Novel Candidates
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text # pip install adjustText for collision-free labels

# 1. Load the Full Statistics Data
# (Assuming the file was saved from Cell 5/6 of your pipeline)
df = pd.read_csv("xgboost_perovskite_discoveries_FullStats.csv")

# Sort by the Worst-Case Risk (Q90) to find the absolute safest economic bets
df_sorted = df.sort_values(by="LCOE_Q90_Worst", ascending=True).reset_index(drop=True)
top_10 = df_sorted.head(10)
the_rest = df_sorted.iloc[10:]

# 2. Setup Publication-Quality Aesthetics
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'sans-serif',
    'axes.linewidth': 1.5,
    'xtick.major.width': 1.5,
    'ytick.major.width': 1.5
})
fig, ax = plt.subplots(figsize=(10, 7))

# 3. Plot the Bulk of the Candidates (Background)
# We use a scatter plot colored by Active Material Cost ($/m^2)
scatter = ax.scatter(
    the_rest['Predicted_Bandgap_eV'], 
    the_rest['LCOE_Q90_Worst'], 
    c=the_rest['Active_Material_Cost_m2'], 
    cmap='plasma', # Plasma provides great contrast for cost mapping
    s=40, 
    alpha=0.6, 
    edgecolors='none',
    label='Screened Candidates'
)

# 4. Highlight the Top 10 "Titanium-Clad" Discoveries
ax.scatter(
    top_10['Predicted_Bandgap_eV'], 
    top_10['LCOE_Q90_Worst'], 
    facecolors='cyan', 
    edgecolors='black', 
    linewidths=1.5,
    s=120, 
    marker='*', 
    zorder=5,
    label='Top 10 "Titanium-Clad" Candidates'
)

# 5. Add Target Reference Lines (Physics & Economics)
# Optimal SQ Peak (~1.34 eV)
ax.axvline(x=1.34, color='gray', linestyle='--', lw=1.5, zorder=1)
ax.text(1.35, ax.get_ylim()[1]*0.95, 'Ideal SQ Peak (1.34 eV)', color='gray', fontsize=10, rotation=270)

# 2030 SunShot Utility Target (~$0.03/kWh) or standard baseline
ax.axhline(y=0.03, color='crimson', linestyle=':', lw=2, zorder=1)
ax.text(ax.get_xlim()[0] + 0.05, 0.031, 'DOE 2030 Target ($0.03/kWh)', color='crimson', fontsize=10, fontweight='bold')

# 6. Add Smart Labels for the Top 10 (Using adjust_text to prevent overlap)
texts = []
for i, row in top_10.iterrows():
    texts.append(ax.text(
        row['Predicted_Bandgap_eV'], 
        row['LCOE_Q90_Worst'], 
        row['Formula'], 
        fontsize=9, 
        fontweight='bold',
        color='black'
    ))

# Repel overlapping labels for a clean, professional look
adjust_text(texts, arrowprops=dict(arrowstyle='->', color='gray', lw=1.0))

# 7. Labels, Legends, and Colorbars
cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label('Active Material Cost ($/m$^2$)', fontweight='bold', fontsize=12)

ax.set_title('Figure 3b: Economic Risk Profiling of 3,208 Unmapped Perovskites\n(Assuming Worst-Case Q90 ML Error Scenarios)', fontweight='bold', fontsize=14)
ax.set_xlabel('XGBoost Predicted Bandgap (eV)', fontweight='bold', fontsize=12)
ax.set_ylabel('Worst-Case LCOE Risk (Q90) ($/kWh)', fontweight='bold', fontsize=12)

# Set logical limits
ax.set_xlim(0.5, 2.5) # The physical PV boundary
ax.set_ylim(0.01, 0.15) # Zoom in on the highly viable economic zone

ax.legend(loc='upper right', frameon=True, edgecolor='black')
ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.show()

# (Optional) Save high-res for publication
# fig.savefig("Figure_3b_Risk_Scatter.png", dpi=300, bbox_inches='tight')
# %%